### Persons in the geographical space

We present various methods of spatialising the birthplaces of our population.

Cf. Sparqlbook [Import birth places](../../sparqlbooks/wdt_import_birth_places.sparqlbook.md)

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from shapely import wkt

import scipy.stats as stats

from geopandas.tools import sjoin
#from geodatasets import get_path

In [ ]:
import networkx as nx
from networkx.algorithms import bipartite

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


import plotly.express as px
import plotly.graph_objects as go

import json

import numpy as np
import seaborn as sns
import math
import os

In [ ]:
import warnings
warnings.filterwarnings('ignore')


In [ ]:
### Librairies déjà installées avec Python
import pprint
import csv


from shutil import copyfile


## Geo data

[Word Regions (ESRI)](https://hub.arcgis.com/datasets/a79a3e4dc55343b08543b1b6133bfb90/explore?location=-0.027457%2C0.000000%2C0.88). For personal use only

In [ ]:
world_filepath = 'geo_data/World_Regions_6144914380456424035.geojson'
world = gpd.read_file(world_filepath)
world.head()

In [ ]:
### Inspect the projection
pprint.pprint(world.crs)

In [ ]:
### Draw the world map

## GeoPandas documentation
# https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.plot.html

# If we want to exclude the Antarctic region
world.clip([-180.0, -10.0, 180.0, 90.0])

ax = world.plot(color="DarkCyan", alpha=0.2, edgecolor="black", figsize=(40,30))

# If we want to exclude the Antarctic region
ax.set_xlim(-170,180)
ax.set_ylim(-65, 90)


plt.show()



## Import the data

The data is available in the da1_data directory and was prepared using the query [documented on this page](../../documentation/wikidata/data-analysis/da1-distribution-of-births-in-time.md)

In [ ]:
## create variable declaring the path to the data
path_to_data = 'da_data/da2-birth-place.csv'

In [ ]:
## creating a Pandas Dataframe (data container in form of a table) 
df_p = pd.read_csv(path_to_data, sep=",", encoding="utf-8", header=0)
df_p.columns = ['uriPer', 'labelPer', 'birthYear', 'gender', 'labelPlace',
                'geometry','uriPlace']

## first rows of the dataframe
df_p.head()

In [ ]:
## Basic infor about the DataFrame
df_p.info()

## Inspect the data


In [ ]:
### Add column names and inspect length of dataframe

print('Number: ',len(df_p))
df_p.iloc[10:13,:]

### Treat multiple birth places

In [ ]:
### Prepare a field that concatenates the values
# for places 
df_p['concat_place'] = df_p.apply(lambda x: (x.labelPlace + ';' + x.uriPlace + ';' + x.geometry), axis=1)

In [ ]:
### Aggregate the places per person

df_pgr=df_p[['uriPer', 'labelPer', 'birthYear', 'concat_place']].groupby(['uriPer', 'labelPer', 'birthYear'],
               as_index=False)\
.agg({'concat_place': '|'.join})
print('Number of persons in dataframe: ', len(df_pgr), '\n---')
df_pgr.head(3)


In [ ]:
### Verify that persons are unique
pers_grouped = df_pgr.groupby('uriPer').size().sort_values(ascending=False)
print('Number of unique persons: ', len(pers_grouped), '\n---')
# print(pers_grouped.describe())
pers_grouped.head()

In [ ]:
### This persons has three birth dates in the orginal data
# this is now cleaned up in the SPARQL query
df_pgr.loc[df_pgr.uriPer.str.contains('Q1243596')]

In [ ]:
df_pgr.loc[df_pgr.uriPer.str.contains('Q117884158')].head()

In [ ]:
### Count the birth places
# The most frequent, first

df_pgr['number'] = df_pgr.concat_place.apply(lambda x : len(x.split('|')))
df_pgr.sort_values("number",ascending=False).head()

In [ ]:
### An example
pprint.pprint(str(df_pgr.loc[df_pgr.uriPer.str.contains('Q1763302')]['concat_place'].to_list()))

In [ ]:
### Number of persons with more then one birth place:
print(len(df_pgr[df_pgr.number > 1]), '\n-----')

### Distribution of multiple places
print(df_pgr.number.describe())



#### Cleaning up and result

In [ ]:
### Take the first of the birth places
df_pgr["first_place"] = df_pgr.concat_place.apply(lambda x: x.split('|')[0])
df_pgr[['labelPlace', 'uriPlace', 'geometry']]=df_pgr.first_place.str.split(';', expand=True)
df_pgr.sort_values("number",ascending=False).head(3)

## Group and map birth places

In [ ]:
### Group and count the number of persons per birth place
p_gr = df_pgr.groupby(by=['uriPlace', 'labelPlace', 'geometry'], as_index=False).size()
p_gr.columns=['uriPlace', 'labelPlace', 'geometry', 'size']
p_gr.sort_values('size', ascending=False).head(10)


In [ ]:
### Try first to convert WKT into Point Geometry
# using a geopandas feature
try:
    p_gr['geometry'] = p_gr['geometry'].apply(wkt.loads)
except Exception as e:
    print(e)

In [ ]:
### there's a wrong value in the data: find it
try:
    df_pgr['len_coord'] = df_pgr.geometry.apply(lambda x: len(x))
except Exception as e:
    print(e)


In [ ]:
df_pgr.sort_values(by='len_coord', ascending=False).head()


In [ ]:
df_pgr.loc[df_pgr.labelPlace=='at sea']

In [ ]:
### Drop the row with the wrong value
# or correct it in the original data
try:
    df_pgr=df_pgr.drop(14790)
except Exception as e:
    print(e)    

In [ ]:
### Re-do grouping after cleaning up
p_gr = df_pgr.groupby(by=['uriPlace', 'labelPlace', 'geometry'], as_index=False).size()
p_gr.columns=['uriPlace', 'labelPlace', 'geometry', 'number']
p_gr.sort_values('number', ascending=False).head(10)


In [ ]:

### Create a dataframe with the POINT geometry
# https://geopandas.org/en/stable/gallery/create_geopandas_from_pandas.html
birth_gdf = gpd.GeoDataFrame(
    p_gr, 
    ### Use this if no conversion before to geometry
    ## Cf. above:  p_gr['geometry'].apply(wkt.loads)
    geometry=gpd.GeoSeries.from_wkt(p_gr.geometry), 
    crs=4326
)

birth_gdf.head(3)



In [ ]:
### Inspect the projection
pprint.pprint(birth_gdf.crs)

In [ ]:
### Draw the world map


ax = world.plot(color="DarkCyan", alpha=0.3, 
                edgecolor="black", linewidth=2, figsize=(80,60))

# If we want to exclude the Antarctic region
ax.set_xlim(-170,180)
ax.set_ylim(-60, 80)

# We can now plot our ``GeoDataFrame``.
birth_gdf.plot(ax=ax, color="red", markersize='number', alpha=0.5)

f_address = "images/birth_places_geopandas.png"
plt.savefig(f_address)
plt.close()

## Use a different projection (Plotly)

In [ ]:
birth_gdf.head(1)

In [ ]:
### Create columns for long / lat
birth_gdf['long'] = birth_gdf['geometry'].x
birth_gdf['lat'] = birth_gdf['geometry'].y
birth_gdf.head(1)

In [ ]:
birth_gdf_plus1 = birth_gdf.loc[birth_gdf.number>1]

In [ ]:
### requires plotly v.5.24 or higher

p_size = [int(s/10+2) for s in birth_gdf_plus1['number']]

fig = px.scatter_map(birth_gdf_plus1, lat="lat", lon="long", size=p_size, 
                     hover_name="labelPlace", hover_data=["number"],
                        color_discrete_sequence=["PowderBlue"],  
                        opacity=0.8, zoom=3.5,
                        center=dict(lon=8.5, lat=47),
                        height=500)
fig.update_layout(
    map_style="white-bg",
    map_layers=[
        {
            "below": 'traces',
            "sourcetype": "raster",
            "sourceattribution": "United States Geological Survey",
            "source": [
                "https://basemap.nationalmap.gov/arcgis/rest/services/USGSImageryOnly/MapServer/tile/{z}/{y}/{x}"
            ]
        }
      ])
#fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
f_address = "interactive_images/birth_places_geological_survey.html"
fig.write_html(f_address)
#fig.show()

In [ ]:
### Open the world regions ESRI file in geojson format (and not dataframe)
with open('../geo_data/esri/World_Regions_6144914380456424035.geojson') as f:
    geojson = json.load(f)

geojson['features'][3]['properties']


In [ ]:
### 

p_size = [int(s/10+2) for s in birth_gdf_plus1['number']]

fig = px.scatter_map(birth_gdf_plus1, lat="lat", lon="long", size=p_size, 
                     hover_name="labelPlace", 
                        color_discrete_sequence=["DodgerBlue"],  zoom=3.5,
                        center=dict(lon=8.5, lat=47),height=600,
                        opacity=0.9)
fig.update_layout(
    map_style="white-bg",
    map_layers=[
        {
            "below": 'traces',
            "sourcetype": "geojson",
            "type":"line",
            "sourceattribution": "ESRI",
            "source": geojson,
            "line": {"width": 0.5},
        }
      ])
f_address = "interactive_images/birth_places_regions.html"
fig.write_html(f_address)
#fig.show()

### Plot the world as a sphere

In [ ]:
### Plot on world as a sphere

lon = birth_gdf["long"]
lat= birth_gdf["lat"]
label= birth_gdf["labelPlace"]
size=[s/10 + 3 for s in birth_gdf["number"]]


# Create the figure
fig = go.Figure(data=go.Scattergeo(
    lon=lon,
    lat=lat,
    mode='markers+text',
    marker=dict(size=size,
                color='red'),
    hovertext=label,
    textposition='top center',
    hoverinfo='text',
    hoverlabel=dict(
        bgcolor='white',
        font_size=12,
        font_family='Arial'
    )
))

# Set the projection to orthogonal
fig.update_layout(
    width=800, 
    height=800,
    geo=dict(
        projection=dict(
            type='orthographic'
        )
    )
)

f_address = "interactive_images/birth_places_globus.html"
fig.write_html(f_address)
# Show the plot
#fig.show()

### Join regions and inspect

We use here spatial joins and associate a region to each place

In [ ]:
birth_gdf.loc[birth_gdf.labelPlace.str.contains('Petersburg') |
              birth_gdf.labelPlace.str.contains('Moscow')
              ].sort_values('number', ascending=False).head(10)

In [ ]:
### GeoPandas spatial join

w_birth_gdf= birth_gdf.sjoin(world)
w_birth_gdf.iloc[[1,5,10]]

In [ ]:
### Number of regions
print(len(world))
world.head(1)

In [ ]:
### Count by region (sum number of each place in the region)
swb = w_birth_gdf.groupby(['FID','REGION'], as_index=False)['number'].sum()
#swb = swb.set_index('FID', drop=False)
print(len(swb))
swb.sort_values('number', ascending=False).head(24)


In [ ]:
### Find places in European Russia with most births 
w_birth_gdf.loc[w_birth_gdf.FID==9].sort_values('number',ascending=False).head(2)

In [ ]:
### Normal Pandas join: add polygons to aggregated regions 
geom_swb=pd.merge(world, swb, left_on='FID', right_on='FID', 
                  how='left')

geom_swb.head(3)

In [ ]:
"""grp_geo = gpd.GeoDataFrame(
    geom_reg_per, 
    crs=4326
)
"""
### simplify geometry 
## https://www.statology.org/how-to-simplify-geographic-data-using-geopandas/
geom_swb["geometry"] = (
    geom_swb.simplify(tolerance=0.5)
)

In [ ]:
max = geom_swb.number.max()

# Create the choropleth map
fig = px.choropleth_map(
    geom_swb,
    geojson=geom_swb.__geo_interface__,
    map_style='white-bg',
    locations=geom_swb.index,
    color='number',  # Replace with your actual column name
    color_continuous_scale='Blues',
    range_color=(0, max),  # Replace with your actual range
    zoom=1,
    hover_name='REGION_x',
    center={'lat': 47, 'lon': 8.5},
    height=600,
    width=800

)

f_address = "interactive_images/birth_places_regions_choropleth.html"
fig.write_html(f_address)
# Show the plot
#fig.show()

## Analyse using periods

In [ ]:
### 30 years periods, but some customisations
lc = [1751, 1811, 1851, 1881, 1911, 1941, 1971, 2001]

In [ ]:
### convert birthYear to integer
df_pgr.birthYear = df_pgr.birthYear.apply(lambda x : int(x))

In [ ]:
try:
    df_pgr = df_pgr.drop('len_coord', axis=1)
except Exception as e:
    print(e)
df_pgr.head(2)

In [ ]:
### fonction pd.cut : https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.cut.html
# On ajoute une nouvelle colonne qui contient la période sur la base de la liste précédente
# et de la valeur de l'année
df_pgr['periods'] = pd.cut(df_pgr['birthYear'], lc, right=False)

### Transformer le code ajouté pour qu'il soit plus lisible
# noter qu'on a arrondi les valeurs
df_pgr['periods'] = df_pgr['periods'].apply(lambda x : str(int(x.left))+'-'+ str(int(x.right)-1))

# Inspection
df_pgr.iloc[[1,5,10]]

In [ ]:
p_gr_per = df_pgr.groupby(by=['uriPlace', 'labelPlace', 'periods', 'geometry'], observed=True, as_index=False).size()
p_gr_per.columns=['uriPlace', 'labelPlace', 'period', 'coordinates', 'size']
#p_grp.geometry = gpd.GeoSeries.from_wkt(p_gr["coordinates"])
p_gr_per.head()


In [ ]:

### Créer un dataframe geopandas avec une colonne contenant une géométrie 
# https://geopandas.org/en/stable/gallery/create_geopandas_from_pandas.html
birth_p_grp = gpd.GeoDataFrame(
    p_gr_per, 
    ### Here we use it
    geometry=gpd.GeoSeries.from_wkt(p_gr_per.coordinates), crs=4326
)

birth_p_grp.head(3)



In [ ]:
### Create columns for long / lat
birth_p_grp['long'] = birth_p_grp['geometry'].x
birth_p_grp['lat'] = birth_p_grp['geometry'].y
birth_p_grp.head()

In [ ]:
# now just plot it on a map with evolution by time
# https://plotly.com/python/animations/
# https://plotly.com/python/scatter-plots-on-maps/

# np.log(s) * 1000
size = [s*5 if s != 0 else 0 for s in birth_p_grp["size"]]




fig = px.scatter_geo(
    birth_p_grp,
    lon="long",
    lat="lat",
    size=size,
    hover_name = "labelPlace",
    animation_frame= "period",
    width=1400, height=600,
    color_discrete_sequence=['red'],
    title="Evolution of birth places through time"
).update_layout(
    # transition_duration=3000,
    #mapbox_style='stamen-watercolor',
    mapbox={"style": "carto-positron", "zoom":2},
    margin={"l": 0, "r": 20, "t": 30, "b": 200}
)



fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 2000
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 20

### Enregistrer l'image, puis l'ouvrir dans un autre onglet du navigateur
f_address = "interactive_images/birth_places_periods.html"
fig.write_html(f_address)
# fig.show()
#plt.close()


### Merge persons and territories

This merge provides regions as additional properties of persons

In [ ]:
with open('../geo_data/esri/World_Regions_6144914380456424035.geojson') as f:
    geojson = json.load(f)

pprint.pprint(geojson['features'][3]['properties'])


In [ ]:
print([f['properties']['FID'] for f in geojson['features']][:3])

In [ ]:
world.head(2)

In [ ]:
### Cleaned up dataframe
df_pgr.iloc[[1,5,10]]

In [ ]:
### create a geo data frame

gdf_pgr = gpd.GeoDataFrame(
    df_pgr, 
    ### Here we use it
    geometry=gpd.GeoSeries.from_wkt(df_pgr.geometry), crs=4326
)

gdf_pgr.head(3)



In [ ]:
### spatial join
pgr_reg = gdf_pgr.sjoin(world)
pgr_reg.iloc[[1,5,10]]

In [ ]:
pgr_reg = pgr_reg.drop(['concat_place', 'number', 'first_place',
                        'index_right', 'SQMI','SQKM'], axis=1)
pgr_reg.head(2)

### Group by regions and periods

In [ ]:
reg_per=pgr_reg.groupby(['FID', 'REGION', 'periods'], observed=True, as_index=False).size()
reg_per.head(3)

In [ ]:
### Merge with world to get polygons
geom_reg_per=pd.merge(reg_per, world, left_on='FID', right_on='FID')
geom_reg_per=geom_reg_per.drop(['REGION_y','SQMI','SQKM'], axis=1)
geom_reg_per.columns=['FID', 'name','periods','number','geometry']
geom_reg_per.head(2)

In [ ]:
### Transform to geoDataFrame
grp_geo = gpd.GeoDataFrame(
    geom_reg_per, 
    crs=4326
)
grp_geo.head(2)

In [ ]:
print(len(grp_geo))

In [ ]:
### Tableau de contingence
X = "REGION"
Y = "periods"  # "0"

pv_per_reg = pgr_reg[[X,Y]].pivot_table(index=Y,columns=X,observed=True, aggfunc=len,margins=True,margins_name="Total").fillna(0).astype(int)
pv_per_reg

In [ ]:
df_t = pv_per_reg.T
df_t

In [ ]:
df_t[df_t.Total > 200]

In [ ]:
df_t[df_t.Total > 200].iloc[:-1, 1:7]

In [ ]:
print(list(df_t.index))

In [ ]:
grp_geo.head(1)

In [ ]:
sel_geo = grp_geo[grp_geo.name.isin(list(df_t.index))].copy(deep=True)
sel_geo.head(2)

In [ ]:
print(grp_geo.crs)

In [ ]:
grp_geo = gpd.GeoDataFrame(
    geom_reg_per, 
    crs=4326
)

### simplify geometry 
## https://www.statology.org/how-to-simplify-geographic-data-using-geopandas/
grp_geo["geometry"] = (
    grp_geo.simplify(tolerance=0.8)
)

In [ ]:
max = grp_geo.number.max()

# Create the choropleth map
fig = px.choropleth_map(
    grp_geo,
    geojson=grp_geo.__geo_interface__,
    locations=grp_geo.index,
    color='number',  # Replace with your actual column name
    color_continuous_scale='Blues',
    range_color=(0, max),  # Replace with your actual range
    zoom=1,
    #map_style='white-bg',
    hover_name='name',
    center={'lat': 47, 'lon': 8.5},
    height=600,
    width=1000,
    animation_frame="periods",
    #title="Temporal evolution of birth regions"

).update_layout(
    # transition_duration=3000,
    #mapbox_style='stamen-watercolor',
    #mapbox={"style": "carto-positron", "zoom":2},
    margin={"l": 0, "r": 20, "t": 30, "b": 200}
)

f_address = "interactive_images/birth_periods_regions_choropleth.html"
fig.write_html(f_address)
# Show the plot
#fig.show()

### Bivariate analysis

In [ ]:
D = df_t[df_t.Total > 200].iloc[:-1, 1:7]

In [ ]:
def bivariee_stats(D, figsize=(8,8)):
    ### Valeurs produites par la fonction de la librairie 'stats'
    statistic, p, dof, expected = stats.chi2_contingency(D)

    n = D.sum().sum()

    print('Chi2 :', statistic.round(2), ', dof :',dof)
    print('p-value :', p)


    print('phi2 = inertie (variance totale) :', statistic/n)


    ### Tableau à l'indépendance
    dfe = round(pd.DataFrame(expected),4)

    ### Coéfficient de Cramer
    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.association.html

    vc = stats.contingency.association(D, method='cramer')
    print('Cramer: ', vc)

    ### Situation à l'indépendance
    indep = pd.DataFrame(expected)
    indep.columns = D.columns
    indep.index = D.index

    ### Résidus pondérés (avec le signe)
    ### Doc. :
    #   Rakotomalala, p.240
    residus_ponderes = (round((D-indep)/np.sqrt(indep),2))
    ### Résidus pondérés
    tableau = residus_ponderes

    fig, ax = plt.subplots(figsize=figsize)         
    # Sample figsize in inches
    g = sns.heatmap(tableau, annot=tableau, cmap="coolwarm", linewidths=.5, ax=ax)
    xlabels = tableau.columns
    px = g.set_xticklabels(xlabels, rotation=60, size=8, 
                           ha='right', rotation_mode='anchor')
    ylabels = tableau.index
    py = g.set_yticklabels(ylabels, rotation=20, size=8)
    
    plt.show()

In [ ]:
## Appliquer la fonction
bivariee_stats(D, figsize=(5,5))